# Cinéfilo: base de conhecimento em RDF, RDFS e OWL

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nicolas-Kleiton/sbc-cinefilo-ontologia/blob/main/Cinefilo_Ontologia.ipynb)

**Mini-Projeto 3, Sistemas Baseados em Conhecimento (UFPB).**

Nos Mini-Projetos 1 e 2, o Cinéfilo recomendava filmes com regras nítidas e com lógica fuzzy. Aqui o
mesmo domínio vira um **grafo de conhecimento**: filmes, profissionais, estúdios e prêmios descritos
em Turtle no arquivo [`cinema.ttl`](cinema.ttl), com hierarquia de classes em RDFS e axiomas em OWL.

O notebook carrega a base com `rdflib` e executa todas as consultas:

| Seção | Conteúdo |
|---|---|
| 1 | Ambiente e funções auxiliares |
| 2 | Carga da base, verificação dos requisitos mínimos e taxonomia |
| 3 | **Parte 1:** 7 consultas com `g.triples()` (T1 a T7) |
| 4 | Inferência com o reasoner OWL-RL e o efeito de cada construção OWL |
| 5 | **Parte 2:** 13 consultas SPARQL (S1 a S13): SELECT, ASK, CONSTRUCT e Update |
| 6 | Resumo das consultas |

Para executar, use **Executar tudo** (Colab) ou **Run All** (Jupyter). As células podem ser
reexecutadas em qualquer ordem: as atualizações da seção 5.4 trabalham sobre uma cópia do grafo.

## 1. Ambiente

A primeira célula instala `rdflib` e `owlrl` quando eles não estão disponíveis (caso do Colab). A
segunda baixa o `cinema.ttl` do GitHub quando o notebook roda fora do repositório clonado.

In [1]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("rdflib") is None or importlib.util.find_spec("owlrl") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rdflib", "owlrl"])

In [2]:
import urllib.request
from pathlib import Path

ARQUIVO_BASE = Path("cinema.ttl")
URL_BASE = "https://raw.githubusercontent.com/Nicolas-Kleiton/sbc-cinefilo-ontologia/main/cinema.ttl"

if not ARQUIVO_BASE.exists():  # no Colab, o .ttl ainda não está no disco
    urllib.request.urlretrieve(URL_BASE, ARQUIVO_BASE)
    print("cinema.ttl baixado do GitHub.")
else:
    print("Usando o cinema.ttl local.")

Usando o cinema.ttl local.


Funções auxiliares usadas no resto do notebook:

- `carregar_base()` lê o `.ttl` e devolve um grafo novo;
- `raciocinar(grafo)` aplica o fechamento dedutivo OWL-RL (`owlrl`) numa **cópia** do grafo;
- `nome(termo, grafo)` mostra o `rdfs:label` de um recurso em vez da URI;
- `mostrar(...)` imprime o resultado de um SELECT como tabela.

In [3]:
from decimal import Decimal

from owlrl import DeductiveClosure, OWLRL_Semantics
from rdflib import Graph, Literal, Namespace
from rdflib.namespace import OWL, RDF, RDFS, XSD

CINE = Namespace("http://ufpb.br/sbc/cinefilo#")
SCHEMA = Namespace("http://schema.org/")
# Vocabulário que o owlrl usa para registrar inconsistências encontradas
ERRO_OWLRL = Namespace("http://www.daml.org/2002/03/agents/agent-ont#")

# Prefixos passados ao rdflib em cada consulta SPARQL (initNs), para não repetir PREFIX.
PREFIXOS = {"cine": CINE, "schema": SCHEMA, "rdf": RDF, "rdfs": RDFS, "owl": OWL, "xsd": XSD}


def carregar_base():
    """Lê cinema.ttl e devolve um grafo com as triplas declaradas."""
    g = Graph()
    g.parse(ARQUIVO_BASE, format="turtle")
    g.bind("cine", CINE)
    g.bind("schema", SCHEMA, replace=True)  # sem replace, o rdflib usaria "schema1:"
    return g


def copiar(grafo):
    """Cópia independente de um grafo, com os mesmos prefixos."""
    novo = Graph()
    for prefixo, ns in grafo.namespaces():
        novo.bind(prefixo, ns, override=True, replace=True)
    novo += grafo
    return novo


def raciocinar(grafo):
    """Devolve uma cópia de `grafo` com todas as triplas deriváveis por OWL-RL.

    O owlrl também gera triplas que não interessam às consultas: literais como
    sujeito (ex.: "8.7" owl:sameAs "8.7") e a identidade reflexiva x owl:sameAs x.
    Elas são removidas para manter o grafo inferido legível.
    """
    inferido = copiar(grafo)
    DeductiveClosure(OWLRL_Semantics).expand(inferido)
    ruido = [t for t in inferido
             if isinstance(t[0], Literal) or (t[1] == OWL.sameAs and t[0] == t[2])]
    for t in ruido:
        inferido.remove(t)
    return inferido


def erros_de_consistencia(grafo_inferido):
    """Mensagens de inconsistência registradas pelo owlrl durante o raciocínio."""
    return sorted(str(m) for m in grafo_inferido.objects(None, ERRO_OWLRL.error))


def nome(termo, grafo):
    """rdfs:label do recurso; se não houver, o nome curto da URI."""
    if termo is None:
        return "—"
    if isinstance(termo, Literal):
        valor = termo.toPython()
        if isinstance(valor, (Decimal, float)):
            return f"{valor:.2f}"
        return str(valor)
    rotulo = grafo.value(termo, RDFS.label)
    return str(rotulo) if rotulo is not None else termo.n3(grafo.namespace_manager)


def nomes(termos, grafo):
    return sorted(nome(t, grafo) for t in termos)


def cabecalho(codigo, proposito):
    print(f"[Consulta {codigo}] {proposito}")
    print("-" * 78)


def mostrar(resultado, grafo):
    """Imprime o resultado de um SELECT como tabela de texto."""
    colunas = [str(v) for v in resultado.vars]
    linhas = [[nome(linha[c], grafo) for c in colunas] for linha in resultado]
    larguras = [max([len(c)] + [len(l[i]) for l in linhas]) for i, c in enumerate(colunas)]
    print("  ".join(c.ljust(w) for c, w in zip(colunas, larguras)))
    print("  ".join("-" * w for w in larguras))
    for linha in linhas:
        print("  ".join(v.ljust(w) for v, w in zip(linha, larguras)))
    print(f"({len(linhas)} linha(s))")

## 2. Carga da base e requisitos mínimos

A base é lida do arquivo Turtle, sem nenhuma tripla criada em código. A célula seguinte confere,
direto no grafo, cada requisito mínimo da especificação.

In [4]:
base = carregar_base()

classes = set(base.subjects(RDF.type, OWL.Class))
props_objeto = set(base.subjects(RDF.type, OWL.ObjectProperty))
props_dados = set(base.subjects(RDF.type, OWL.DatatypeProperty))
instancias = {s for s, o in base.subject_objects(RDF.type) if o in classes}

# Construções OWL: predicados do vocabulário owl: (inverseOf, disjointWith...) e
# características de propriedade (TransitiveProperty...). As declarações básicas
# (owl:Class, owl:ObjectProperty...) não contam como construção.
declaracoes = {OWL.Class, OWL.ObjectProperty, OWL.DatatypeProperty, OWL.Ontology}
construcoes_owl = {p for p in base.predicates() if p.startswith(OWL)}
construcoes_owl |= {o for o in base.objects(None, RDF.type) if o.startswith(OWL) and o not in declaracoes}

props_com_dominio_e_range = {p for p in props_objeto | props_dados
                             if (p, RDFS.domain, None) in base and (p, RDFS.range, None) in base}


def profundidade(classe):
    """Número de níveis entre a classe e a raiz da sua hierarquia."""
    pais = list(base.objects(classe, RDFS.subClassOf))
    return 0 if not pais else 1 + max(profundidade(p) for p in pais)


requisitos = [
    ("Classes", len(classes), 8),
    ("Níveis abaixo da raiz", max(profundidade(c) for c in classes), 2),
    ("Propriedades de objeto", len(props_objeto), 5),
    ("Propriedades de dados", len(props_dados), 5),
    ("Propriedades (total)", len(props_objeto | props_dados), 10),
    ("Instâncias", len(instancias), 25),
    ("Triplas", len(base), 50),
    ("Construções OWL", len(construcoes_owl), 5),
]
print(f"{'Requisito':<25}{'Na base':>9}{'Mínimo':>9}")
for rotulo, valor, minimo in requisitos:
    print(f"{rotulo:<25}{valor:>9}{minimo:>9}  {'ok' if valor >= minimo else 'FALTA'}")

print(f"\nPropriedades com domain e range: {len(props_com_dominio_e_range)} de {len(props_objeto | props_dados)}")
print("Construções OWL usadas:", ", ".join(sorted(c.n3(base.namespace_manager) for c in construcoes_owl)))

Requisito                  Na base   Mínimo
Classes                         18        8  ok
Níveis abaixo da raiz            3        2  ok
Propriedades de objeto          12        5  ok
Propriedades de dados            7        5  ok
Propriedades (total)            19       10  ok
Instâncias                      68       25  ok
Triplas                        561       50  ok
Construções OWL                  7        5  ok

Propriedades com domain e range: 19 de 19
Construções OWL usadas: owl:AsymmetricProperty, owl:FunctionalProperty, owl:SymmetricProperty, owl:TransitiveProperty, owl:disjointWith, owl:equivalentClass, owl:inverseOf


### Taxonomia

A árvore abaixo é gerada a partir das triplas `rdfs:subClassOf` do arquivo. São quatro raízes
independentes (Obra, Pessoa, Organizacao e Premio). A hierarquia mais profunda é a de filmes, com
três níveis abaixo de `Obra`.

In [5]:
def imprimir_arvore(classe, prefixo="", ultimo=True, raiz=True):
    rotulo = classe.n3(base.namespace_manager)
    n = sum(1 for _ in base.subjects(RDF.type, classe))
    sufixo = f"  ({n} instância(s) declarada(s))" if n else ""
    print((rotulo if raiz else prefixo + ("└─ " if ultimo else "├─ ") + rotulo) + sufixo)
    filhos = sorted(base.subjects(RDFS.subClassOf, classe))
    novo_prefixo = "" if raiz else prefixo + ("   " if ultimo else "│  ")
    for i, filho in enumerate(filhos):
        imprimir_arvore(filho, novo_prefixo, i == len(filhos) - 1, raiz=False)


raizes = sorted(c for c in classes if (c, RDFS.subClassOf, None) not in base)
for r in raizes:
    imprimir_arvore(r)

cine:Obra
└─ cine:Filme
   ├─ cine:Documentario  (1 instância(s) declarada(s))
   └─ cine:Ficcao
      ├─ cine:Acao  (2 instância(s) declarada(s))
      ├─ cine:Animacao  (3 instância(s) declarada(s))
      ├─ cine:Drama  (6 instância(s) declarada(s))
      ├─ cine:FiccaoCientifica  (7 instância(s) declarada(s))
      └─ cine:Terror  (2 instância(s) declarada(s))
cine:Organizacao
└─ cine:Estudio  (8 instância(s) declarada(s))
cine:Pessoa
└─ cine:Profissional
   ├─ cine:Ator  (19 instância(s) declarada(s))
   ├─ cine:Compositor  (4 instância(s) declarada(s))
   ├─ cine:Diretor  (10 instância(s) declarada(s))
   └─ cine:Roteirista  (3 instância(s) declarada(s))
cine:Premio  (7 instância(s) declarada(s))


## 3. Parte 1: consultas com `g.triples()`

`g.triples((s, p, o))` devolve as triplas que casam com o padrão, e `None` funciona como coringa.
As consultas desta seção rodam sobre o grafo **sem inferência** (`base`), ou seja, só sobre o que
está escrito no `.ttl`. Cada uma usa um padrão diferente:

| Consulta | Padrão | Pergunta |
|---|---|---|
| T1 | `(s, p, None)` | Quem está no elenco de Interestelar? |
| T2 | `(None, p, o)` | Que filmes Villeneuve dirigiu? |
| T3 | `(None, p, None)` | Quais são as maiores notas do IMDb? |
| T4 | `(s, None, None)` | O que a base diz sobre Clint Eastwood? |
| T5 | `(None, None, o)` | Onde Clint Eastwood aparece como objeto? |
| T6 | `(s, None, o)` | Que relação liga Nolan a Villeneuve? |
| T7 | `(None, rdf:type, o)` | Que filmes são do tipo `Ficcao`? |

In [6]:
# T1: padrão (sujeito fixo, predicado fixo, None)
# Busca:    os atores ligados a Interestelar por cine:temNoElenco.
# Esperado: Anne Hathaway, Jessica Chastain e Matthew McConaughey.
cabecalho("T1", "Elenco de Interestelar  (s, p, None)")
for ator in nomes((o for _, _, o in base.triples((CINE.interestelar, CINE.temNoElenco, None))), base):
    print(" ", ator)

[Consulta T1] Elenco de Interestelar  (s, p, None)
------------------------------------------------------------------------------
  Anne Hathaway
  Jessica Chastain
  Matthew McConaughey


In [7]:
# T2: padrão (None, predicado fixo, objeto fixo)
# Busca:    os filmes cujo cine:dirigidoPor é Denis Villeneuve.
# Esperado: A Chegada, Blade Runner 2049, Duna e Duna: Parte Dois.
cabecalho("T2", "Filmes dirigidos por Denis Villeneuve  (None, p, o)")
for titulo in nomes((s for s, _, _ in base.triples((None, CINE.dirigidoPor, CINE.villeneuve))), base):
    print(" ", titulo)

[Consulta T2] Filmes dirigidos por Denis Villeneuve  (None, p, o)
------------------------------------------------------------------------------
  A Chegada
  Blade Runner 2049
  Duna
  Duna: Parte Dois


In [8]:
# T3: padrão (None, predicado fixo, None)
# Busca:    todas as triplas cine:notaIMDb, ou seja, a nota de cada filme. Depois
#           ordena em Python e mostra as cinco maiores.
# Esperado: 17 notas; no topo, Batman: O Cavaleiro das Trevas (9.0), A Origem (8.8),
#           Interestelar (8.7), Duna: Parte Dois (8.5) e Parasita (8.5).
cabecalho("T3", "Cinco maiores notas no IMDb  (None, p, None)")
notas = [(o.toPython(), nome(s, base)) for s, _, o in base.triples((None, CINE.notaIMDb, None))]
print(f"  {len(notas)} filmes com nota registrada")
for nota, titulo in sorted(notas, key=lambda x: (-x[0], x[1]))[:5]:
    print(f"  {nota:>4}  {titulo}")

[Consulta T3] Cinco maiores notas no IMDb  (None, p, None)
------------------------------------------------------------------------------
  17 filmes com nota registrada
   9.0  Batman: O Cavaleiro das Trevas
   8.8  A Origem
   8.7  Interestelar
   8.5  Duna: Parte Dois
   8.5  Parasita


In [9]:
# T4: padrão (sujeito fixo, None, None)
# Busca:    todas as triplas em que Clint Eastwood é o sujeito (o "perfil" dele).
# Esperado: tipo Diretor, nome, ano de nascimento 1930, nacionalidade EUA e a
#           colaboração com Morgan Freeman. Nada diz que ele é ator: esse fato está
#           em triplas onde ele aparece como objeto (consulta T5).
cabecalho("T4", "Tudo o que é afirmado sobre Clint Eastwood  (s, None, None)")
for s, p, o in sorted(base.triples((CINE.eastwood, None, None))):
    print(f"  {p.n3(base.namespace_manager):<22} {nome(o, base)}")

[Consulta T4] Tudo o que é afirmado sobre Clint Eastwood  (s, None, None)
------------------------------------------------------------------------------
  cine:anoNascimento     1930
  cine:colaborouCom      Morgan Freeman
  cine:nacionalidade     EUA
  rdf:type               Diretor
  rdfs:label             Clint Eastwood


In [10]:
# T5: padrão (None, None, objeto fixo)
# Busca:    todas as triplas em que Clint Eastwood é o objeto.
# Esperado: Menina de Ouro aponta para ele duas vezes, por cine:dirigidoPor e por
#           cine:temNoElenco: ele dirigiu e atuou no mesmo filme.
cabecalho("T5", "Onde Clint Eastwood aparece como objeto  (None, None, o)")
for s, p, o in sorted(base.triples((None, None, CINE.eastwood))):
    print(f"  {nome(s, base):<16} {p.n3(base.namespace_manager)}")

[Consulta T5] Onde Clint Eastwood aparece como objeto  (None, None, o)
------------------------------------------------------------------------------
  Menina de Ouro   cine:dirigidoPor
  Menina de Ouro   cine:temNoElenco


In [11]:
# T6: padrão (sujeito fixo, None, objeto fixo)
# Busca:    qualquer predicado que ligue Nolan diretamente a Villeneuve.
# Esperado: uma única tripla, cine:influenciou.
cabecalho("T6", "Que relação liga Christopher Nolan a Denis Villeneuve  (s, None, o)")
for s, p, o in base.triples((CINE.nolan, None, CINE.villeneuve)):
    print(" ", p.n3(base.namespace_manager))

[Consulta T6] Que relação liga Christopher Nolan a Denis Villeneuve  (s, None, o)
------------------------------------------------------------------------------
  cine:influenciou


In [12]:
# T7: padrão (None, rdf:type, objeto fixo)
# Busca:    as instâncias declaradas como cine:Ficcao.
# Esperado: NENHUMA. Os filmes declaram só o gênero mais específico (Drama, Terror...).
#           Que um Drama também é Ficcao depende de rdfs:subClassOf, e g.triples() só
#           enxerga o que está escrito. Na seção 4, o reasoner resolve isso.
cabecalho("T7", "Filmes do tipo Ficcao, sem inferência  (None, rdf:type, o)")
ficcao = [s for s, _, _ in base.triples((None, RDF.type, CINE.Ficcao))]
print(f"  {len(ficcao)} resultado(s)")

[Consulta T7] Filmes do tipo Ficcao, sem inferência  (None, rdf:type, o)
------------------------------------------------------------------------------
  0 resultado(s)


## 4. Inferência: o que o RDFS e o OWL acrescentam

Como mostra a consulta T7, o grafo carregado só contém o que foi escrito. O reasoner
`owlrl` aplica as regras de RDFS e OWL-RL até o ponto fixo, quando nenhuma regra gera tripla nova,
e adiciona ao grafo tudo o que pode ser deduzido. É o `DeductiveClosure(OWLRL_Semantics).expand(g)`
da aula, aplicado aqui numa cópia para manter a base original intacta.

In [13]:
inferida = raciocinar(base)

print(f"Triplas declaradas: {len(base)}")
print(f"Triplas após o raciocínio: {len(inferida)}  (+{len(inferida) - len(base)} inferidas)")
print("Inconsistências encontradas:", erros_de_consistencia(inferida) or "nenhuma")

Triplas declaradas: 561
Triplas após o raciocínio: 1169  (+608 inferidas)
Inconsistências encontradas: nenhuma


A célula abaixo repete o mesmo padrão de `g.triples()` antes e depois do raciocínio, um para cada
construção da base. Tudo o que aparece em **depois** e não em **antes** foi deduzido, e nenhuma
dessas triplas está escrita no `.ttl`.

In [14]:
def comparar(construcao, pergunta, padrao, posicao):
    """Mostra o resultado de um padrão g.triples() na base e no grafo inferido."""
    antes = nomes((t[posicao] for t in base.triples(padrao)), base)
    depois = nomes((t[posicao] for t in inferida.triples(padrao)), inferida)
    novos = [x for x in depois if x not in antes]
    print(f"{construcao}: {pergunta}")
    print(f"  antes  ({len(antes):>2}): {', '.join(antes) or '—'}")
    print(f"  depois ({len(depois):>2}): {', '.join(depois)}")
    print(f"  inferidos: {', '.join(novos) or '—'}\n")


comparar("rdfs:subClassOf", "que filmes são Ficcao?",
         (None, RDF.type, CINE.Ficcao), 0)
comparar("rdfs:domain / rdfs:range", "quais são os tipos de Christopher Nolan?",
         (CINE.nolan, RDF.type, None), 2)
comparar("owl:inverseOf", "o que Christopher Nolan dirigiu?",
         (CINE.nolan, CINE.dirigiu, None), 2)
comparar("owl:TransitiveProperty", "quem Stanley Kubrick influenciou?",
         (CINE.kubrick, CINE.influenciou, None), 2)
comparar("owl:SymmetricProperty", "com quem Hans Zimmer colaborou?",
         (CINE.zimmer, CINE.colaborouCom, None), 2)
comparar("rdfs:subPropertyOf + owl:TransitiveProperty", "Toy Story 3 vem depois de quais filmes?",
         (CINE.toyStory3, CINE.posteriorNaFranquiaA, None), 2)
comparar("owl:equivalentClass", "que recursos são schema:Movie?",
         (None, RDF.type, SCHEMA.Movie), 0)

rdfs:subClassOf: que filmes são Ficcao?
  antes  ( 0): —
  depois (16): 2001: Uma Odisseia no Espaço, A Chegada, A Origem, Batman: O Cavaleiro das Trevas, Blade Runner 2049, Corra!, Duna, Duna: Parte Dois, Interestelar, Menina de Ouro, O Iluminado, Oppenheimer, Parasita, Toy Story, Toy Story 2, Toy Story 3
  inferidos: 2001: Uma Odisseia no Espaço, A Chegada, A Origem, Batman: O Cavaleiro das Trevas, Blade Runner 2049, Corra!, Duna, Duna: Parte Dois, Interestelar, Menina de Ouro, O Iluminado, Oppenheimer, Parasita, Toy Story, Toy Story 2, Toy Story 3

rdfs:domain / rdfs:range: quais são os tipos de Christopher Nolan?
  antes  ( 1): Diretor
  depois ( 6): Diretor, Pessoa, Profissional de cinema, Roteirista, owl:Thing, schema:Person
  inferidos: Pessoa, Profissional de cinema, Roteirista, owl:Thing, schema:Person

owl:inverseOf: o que Christopher Nolan dirigiu?
  antes  ( 0): —
  depois ( 4): A Origem, Batman: O Cavaleiro das Trevas, Interestelar, Oppenheimer
  inferidos: A Origem, Batma

Leitura do resultado:

- **subClassOf:** os 16 filmes de ficção agora são `Ficcao`. Só *Free Solo* fica de fora, por ser
  documentário.
- **domain/range:** Nolan foi declarado só como `Diretor`. Como ele aparece em `roteirizadoPor`,
  cujo `rdfs:range` é `Roteirista`, o reasoner deduz que ele também é roteirista. A partir daí, a
  hierarquia o torna `Profissional` e `Pessoa`, e a equivalência o torna `schema:Person`.
- **inverseOf:** a base registra `filme dirigidoPor pessoa`. A pergunta feita a partir da pessoa
  passa a ter resposta sem duplicar dados.
- **TransitiveProperty:** Kubrick → Nolan → Villeneuve fecha em Kubrick → Villeneuve.
- **SymmetricProperty:** a colaboração foi escrita a partir de Nolan e de Villeneuve, e passa a valer
  também a partir de Zimmer.
- **subPropertyOf + TransitiveProperty:** `sequenciaDe` (direta) implica `posteriorNaFranquiaA`, que é
  transitiva. Toy Story 3 é a sequência direta só de Toy Story 2, mas vem depois dos dois.
- **equivalentClass:** quem consulta pelo vocabulário schema.org encontra os filmes da base.

### 4.1 Disjunção e assimetria detectam erros

`owl:disjointWith` e `owl:AsymmetricProperty` não geram fatos novos: elas proíbem combinações. Para
mostrar que têm efeito, a célula abaixo insere dois erros numa cópia da base e roda o reasoner:

1. *Free Solo* (um `Documentario`) é marcado também como `Drama`. Como `Drama` é subclasse de
   `Ficcao`, e `Ficcao` é disjunta de `Documentario`, isso é uma contradição.
2. Toy Story passa a ser sequência de Toy Story 2, mas Toy Story 2 já é sequência de Toy Story.

In [15]:
com_erros = copiar(base)
com_erros.add((CINE.freeSolo, RDF.type, CINE.Drama))
com_erros.add((CINE.toyStory, CINE.sequenciaDe, CINE.toyStory2))

for mensagem in erros_de_consistencia(raciocinar(com_erros)):
    print("-", mensagem.replace(str(CINE), "cine:"))

- Disjoint classes cine:Ficcao and cine:Documentario have a common individual cine:freeSolo
- Erroneous usage of asymmetric property cine:sequenciaDe on cine:toyStory and cine:toyStory2
- Erroneous usage of asymmetric property cine:sequenciaDe on cine:toyStory2 and cine:toyStory


> **Limitação observada:** o owlrl não acusa conflito de `owl:FunctionalProperty` entre dois
> *literais* (por exemplo, duas notas diferentes para o mesmo filme). O OWL trabalha com mundo aberto
> e não foi feito para validação; como diz a aula, esse papel é do SHACL. Por isso, a consulta S12
> confere a regra "uma nota por filme" com SPARQL.

## 5. Parte 2: consultas SPARQL

As consultas rodam sobre o grafo **inferido**, para que possam usar as triplas deduzidas (tipos
herdados, inversas, transitividade). Os prefixos `cine:`, `rdfs:`, `schema:` etc. são passados por
`initNs`, por isso não aparecem no texto de cada consulta.

| Consulta | Forma | Recurso SPARQL |
|---|---|---|
| S1 | SELECT | `FILTER` com duas condições |
| S2 | SELECT | `ORDER BY` + `LIMIT` |
| S3 | SELECT | agregação `COUNT`/`AVG`/`MIN`/`MAX` + `GROUP BY` + `HAVING` |
| S4 | SELECT | agregação por gênero (tipos inferidos) |
| S5 | SELECT | `VALUES` + `GROUP_CONCAT` + `HAVING` |
| S6 | SELECT | `OPTIONAL` |
| S7 | SELECT | caminho de propriedade `influenciou+` |
| S8, S9 | ASK | — |
| S10 | CONSTRUCT | exportação para schema.org |
| S11 | INSERT DATA | — |
| S12 | DELETE/INSERT | — |
| S13 | DELETE | — |

### 5.1 SELECT

In [16]:
# S1: SELECT com FILTER
# Ficção de qualquer subgênero (cine:Ficcao só existe por inferência), lançada a
# partir de 2010 e com nota a partir de 8.3. Esperado: 6 filmes.
cabecalho("S1", "Ficção lançada a partir de 2010 com nota ≥ 8.3")
mostrar(inferida.query("""
    SELECT ?filme ?ano ?nota ?diretor
    WHERE {
        ?filme a cine:Ficcao ;
               cine:anoLancamento ?ano ;
               cine:notaIMDb ?nota ;
               cine:dirigidoPor ?diretor .
        FILTER (?ano >= 2010 && ?nota >= 8.3)
    }
    ORDER BY DESC(?nota) ?ano
""", initNs=PREFIXOS), inferida)

[Consulta S1] Ficção lançada a partir de 2010 com nota ≥ 8.3
------------------------------------------------------------------------------


filme             ano   nota  diretor          
----------------  ----  ----  -----------------
A Origem          2010  8.80  Christopher Nolan
Interestelar      2014  8.70  Christopher Nolan
Parasita          2019  8.50  Bong Joon-ho     
Duna: Parte Dois  2024  8.50  Denis Villeneuve 
Toy Story 3       2010  8.30  Lee Unkrich      
Oppenheimer       2023  8.30  Christopher Nolan
(6 linha(s))


In [17]:
# S2: SELECT com ORDER BY e LIMIT
# Para quem tem pouco tempo: os cinco filmes mais longos. Esperado: Oppenheimer
# (180 min) no topo, seguido de Interestelar (169).
cabecalho("S2", "Os 5 filmes mais longos")
mostrar(inferida.query("""
    SELECT ?filme ?duracao ?ano
    WHERE {
        ?filme a cine:Filme ;
               cine:duracaoMinutos ?duracao ;
               cine:anoLancamento ?ano .
    }
    ORDER BY DESC(?duracao)
    LIMIT 5
""", initNs=PREFIXOS), inferida)

[Consulta S2] Os 5 filmes mais longos
------------------------------------------------------------------------------
filme              duracao  ano 
-----------------  -------  ----
Oppenheimer        180      2023
Interestelar       169      2014
Duna: Parte Dois   166      2024
Blade Runner 2049  164      2017
Duna               155      2021
(5 linha(s))


In [18]:
# S3: SELECT com agregação (COUNT, AVG, MIN, MAX), GROUP BY e HAVING
# Usa cine:dirigiu, que não está escrita na base: vem da inversa de cine:dirigidoPor.
# Esperado: Nolan e Villeneuve com 4 filmes cada; Nolan com a maior média (8.70).
cabecalho("S3", "Diretores com 2+ filmes: quantidade, nota média e período de atividade")
mostrar(inferida.query("""
    SELECT ?diretor (COUNT(?filme) AS ?filmes) (AVG(?nota) AS ?media)
           (MIN(?ano) AS ?primeiro) (MAX(?ano) AS ?ultimo)
    WHERE {
        ?diretor cine:dirigiu ?filme .
        ?filme cine:notaIMDb ?nota ;
               cine:anoLancamento ?ano .
    }
    GROUP BY ?diretor
    HAVING (COUNT(?filme) >= 2)
    ORDER BY DESC(?media) DESC(?filmes) ?diretor
""", initNs=PREFIXOS), inferida)

[Consulta S3] Diretores com 2+ filmes: quantidade, nota média e período de atividade
------------------------------------------------------------------------------
diretor            filmes  media  primeiro  ultimo
-----------------  ------  -----  --------  ------
Christopher Nolan  4       8.70   2008      2023  
Stanley Kubrick    2       8.35   1968      1980  
Denis Villeneuve   4       8.10   2016      2024  
John Lasseter      2       8.10   1995      1999  
Lee Unkrich        2       8.10   1999      2010  
(5 linha(s))


In [19]:
# S4: SELECT com agregação por gênero
# Cada filme declara só o gênero; a subclasse de Ficcao vem da taxonomia. Um filme
# pode contar em mais de um gênero (Interestelar é Ficção científica e Drama).
# owl:Nothing é excluída porque, por definição, é subclasse de tudo.
# Esperado: Ficção científica com 7 filmes, Drama com 6.
cabecalho("S4", "Filmes e nota média por gênero de ficção")
mostrar(inferida.query("""
    SELECT ?genero (COUNT(?filme) AS ?filmes) (AVG(?nota) AS ?media)
    WHERE {
        ?genero rdfs:subClassOf cine:Ficcao .
        FILTER (?genero NOT IN (cine:Ficcao, owl:Nothing))
        ?filme a ?genero ;
               cine:notaIMDb ?nota .
    }
    GROUP BY ?genero
    ORDER BY DESC(?filmes) DESC(?media)
""", initNs=PREFIXOS), inferida)

[Consulta S4] Filmes e nota média por gênero de ficção
------------------------------------------------------------------------------
genero             filmes  media
-----------------  ------  -----
Ficção científica  7       8.31 
Drama              6       8.42 
Animação           3       8.17 
Ação               2       8.90 
Terror             2       8.10 
(5 linha(s))


In [20]:
# S5: SELECT com VALUES, GROUP_CONCAT e HAVING
# Pessoas com mais de um papel profissional. Nenhuma foi declarada com dois papéis:
# o segundo vem do rdfs:range de roteirizadoPor (Roteirista) e de temNoElenco (Ator).
# Esperado: Bong Joon-ho, Christopher Nolan, Jordan Peele e Stanley Kubrick como
# diretores-roteiristas, e Clint Eastwood como diretor e ator.
cabecalho("S5", "Profissionais que acumulam mais de um papel")
mostrar(inferida.query("""
    SELECT (SAMPLE(?nome) AS ?pessoa) (GROUP_CONCAT(?papel; separator=" + ") AS ?papeis)
    WHERE {
        VALUES ?classe { cine:Diretor cine:Ator cine:Roteirista cine:Compositor }
        ?p a ?classe ;
           rdfs:label ?nome .
        ?classe rdfs:label ?papel .
    }
    GROUP BY ?p
    HAVING (COUNT(?classe) > 1)
    ORDER BY ?pessoa
""", initNs=PREFIXOS), inferida)

[Consulta S5] Profissionais que acumulam mais de um papel
------------------------------------------------------------------------------
pessoa             papeis              
-----------------  --------------------
Bong Joon-ho       Diretor + Roteirista
Christopher Nolan  Diretor + Roteirista
Clint Eastwood     Diretor + Ator      
Jordan Peele       Diretor + Roteirista
Stanley Kubrick    Diretor + Roteirista
(5 linha(s))


In [21]:
# S6: SELECT com OPTIONAL
# Todos os filmes de ficção científica, com os prêmios quando houver. Sem o OPTIONAL,
# A Chegada (que não ganhou nenhum prêmio registrado) sumiria do resultado.
# Esperado: 7 filmes; A Chegada com "—" na coluna de prêmio.
cabecalho("S6", "Ficção científica e prêmios recebidos (quando houver)")
mostrar(inferida.query("""
    SELECT ?filme ?ano ?premio
    WHERE {
        ?filme a cine:FiccaoCientifica ;
               cine:anoLancamento ?ano .
        OPTIONAL { ?filme cine:ganhouPremio ?premio . }
    }
    ORDER BY ?ano ?premio
""", initNs=PREFIXOS), inferida)

[Consulta S6] Ficção científica e prêmios recebidos (quando houver)
------------------------------------------------------------------------------
filme                         ano   premio                           
----------------------------  ----  ---------------------------------
2001: Uma Odisseia no Espaço  1968  Oscar de Melhores Efeitos Visuais
A Origem                      2010  Oscar de Melhores Efeitos Visuais
Interestelar                  2014  Oscar de Melhores Efeitos Visuais
A Chegada                     2016  —                                
Blade Runner 2049             2017  Oscar de Melhores Efeitos Visuais
Duna                          2021  Oscar de Melhores Efeitos Visuais
Duna                          2021  Oscar de Melhor Trilha Sonora    
Duna: Parte Dois              2024  Oscar de Melhores Efeitos Visuais
(8 linha(s))


In [22]:
# S7: SELECT com caminho de propriedade (influenciou+)
# Roda sobre a BASE SEM INFERÊNCIA. O operador + percorre a relação por um ou mais
# saltos, então a consulta chega à cadeia de influência completa sem o reasoner.
# Esperado: Jordan Peele, Christopher Nolan e Denis Villeneuve.
cabecalho("S7", "Diretores influenciados por Kubrick, direta ou indiretamente (base sem inferência)")
mostrar(base.query("""
    SELECT ?influenciado ?nascimento
    WHERE {
        cine:kubrick cine:influenciou+ ?influenciado .
        ?influenciado cine:anoNascimento ?nascimento .
    }
    ORDER BY ?nascimento
""", initNs=PREFIXOS), base)

[Consulta S7] Diretores influenciados por Kubrick, direta ou indiretamente (base sem inferência)
------------------------------------------------------------------------------
influenciado       nascimento
-----------------  ----------
Denis Villeneuve   1967      
Christopher Nolan  1970      
Jordan Peele       1979      
(3 linha(s))


### 5.2 ASK

ASK responde apenas `True` ou `False`, conforme o padrão tenha ou não alguma solução no grafo.

In [23]:
# S8: ASK, a mesma pergunta antes e depois da inferência
# Esperado: False na base (não há tripla kubrick influenciou villeneuve escrita) e
#           True no grafo inferido (owl:TransitiveProperty).
cabecalho("S8", "Stanley Kubrick influenciou Denis Villeneuve?")
pergunta = "ASK { cine:kubrick cine:influenciou cine:villeneuve }"
print("  base sem inferência:", base.query(pergunta, initNs=PREFIXOS).askAnswer)
print("  grafo inferido:     ", inferida.query(pergunta, initNs=PREFIXOS).askAnswer)

[Consulta S8] Stanley Kubrick influenciou Denis Villeneuve?
------------------------------------------------------------------------------
  base sem inferência: False
  grafo inferido:      True


In [24]:
# S9: ASK com junção sobre propriedades inferidas
# Alguém atuou num filme que também dirigiu? Usa dirigiu e atuouEm, as duas inversas.
# Esperado: True (Clint Eastwood em Menina de Ouro).
cabecalho("S9", "Existe alguém que atuou em um filme que ele mesmo dirigiu?")
print(" ", inferida.query("""
    ASK {
        ?pessoa cine:dirigiu ?filme ;
                cine:atuouEm ?filme .
    }
""", initNs=PREFIXOS).askAnswer)

[Consulta S9] Existe alguém que atuou em um filme que ele mesmo dirigiu?
------------------------------------------------------------------------------
  True


### 5.3 CONSTRUCT

CONSTRUCT não devolve uma tabela: devolve um **grafo novo**, montado a partir de um molde. Aqui ele
exporta o catálogo para o vocabulário [schema.org](https://schema.org/Movie), o mesmo que buscadores
leem. É a aplicação prática do alinhamento `cine:Filme ≡ schema:Movie`. A duração é convertida para
o formato ISO 8601 que o schema.org espera (`PT169M`).

In [25]:
# S10: CONSTRUCT
# Esperado: um grafo com os 17 filmes descritos só com termos schema: (name,
# datePublished, duration, director, actor), mais as pessoas como schema:Person.
cabecalho("S10", "Exportação do catálogo para o vocabulário schema.org")
exportado = inferida.query("""
    CONSTRUCT {
        ?filme a schema:Movie ;
               schema:name ?titulo ;
               schema:datePublished ?ano ;
               schema:duration ?duracao ;
               schema:director ?diretor ;
               schema:actor ?ator .
        ?diretor a schema:Person ; schema:name ?nomeDiretor .
        ?ator a schema:Person ; schema:name ?nomeAtor .
    }
    WHERE {
        ?filme a cine:Filme ;
               rdfs:label ?titulo ;
               cine:anoLancamento ?ano ;
               cine:duracaoMinutos ?minutos ;
               cine:dirigidoPor ?diretor .
        ?diretor rdfs:label ?nomeDiretor .
        BIND (CONCAT("PT", STR(?minutos), "M") AS ?duracao)
        OPTIONAL { ?filme cine:temNoElenco ?ator . ?ator rdfs:label ?nomeAtor . }
    }
""", initNs=PREFIXOS).graph

exportado.bind("schema", SCHEMA, replace=True)
exportado.bind("cine", CINE)
filmes_exportados = set(exportado.subjects(RDF.type, SCHEMA.Movie))
print(f"Grafo construído: {len(exportado)} triplas, {len(filmes_exportados)} filmes.")
print("Usa só termos schema:?",
      all(p == RDF.type or p.startswith(SCHEMA) for p in exportado.predicates()))

# Recorte do grafo exportado: Interestelar e as pessoas ligadas a ele
recorte = Graph()
recorte.bind("schema", SCHEMA, replace=True)
recorte.bind("cine", CINE)
recorte += exportado.triples((CINE.interestelar, None, None))
for pessoa in exportado.objects(CINE.interestelar, None):
    recorte += exportado.triples((pessoa, None, None))
print("\nRecorte (Interestelar):\n")
print(recorte.serialize(format="turtle"))

[Consulta S10] Exportação do catálogo para o vocabulário schema.org
------------------------------------------------------------------------------


Grafo construído: 172 triplas, 17 filmes.
Usa só termos schema:? True

Recorte (Interestelar):

@prefix cine: <http://ufpb.br/sbc/cinefilo#> .
@prefix schema: <http://schema.org/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

cine:interestelar a schema:Movie ;
    schema:actor cine:chastain,
        cine:hathaway,
        cine:mcconaughey ;
    schema:datePublished 2014 ;
    schema:director cine:nolan ;
    schema:duration "PT169M" ;
    schema:name "Interestelar" .

cine:chastain a schema:Person ;
    schema:name "Jessica Chastain" .

cine:hathaway a schema:Person ;
    schema:name "Anne Hathaway" .

cine:mcconaughey a schema:Person ;
    schema:name "Matthew McConaughey" .

cine:nolan a schema:Person ;
    schema:name "Christopher Nolan" .


